# Kaggle - Frames para anime (imagem a imagem)

Sessao: **GPU T4 x2** (ou P100), **Internet: ON**.

Fluxo: video -> frames (ffmpeg) -> ComfyUI transforma cada frame (mesma seed) -> frames de saida -> video.

1. Suba seu video como Dataset (Add Input -> Upload) ou coloque em `/kaggle/working`.
2. Suba tambem `workflows/img2img_frame_api.json` para `/kaggle/working/`.
3. Rode as celulas em ordem. As celulas 4 a 6 podem ser repetidas quantas vezes quiser.
4. Para editar frames (apagar nome, desenhar a bola): baixe `/kaggle/working/frames_out`, edite mantendo o nome (`0007.png`), e coloque os editados em `/kaggle/working/frames_edited` antes da celula 6.


In [ ]:
%cd /kaggle/working
![ ! -d ComfyUI ] && git clone https://github.com/comfyanonymous/ComfyUI
%cd /kaggle/working/ComfyUI
!git pull
!pip install -q -r requirements.txt
!pip install -q GitPython

%cd /kaggle/working/ComfyUI/custom_nodes
![ ! -d comfyui_controlnet_aux ] && git clone https://github.com/Fannovel16/comfyui_controlnet_aux
!pip install -q -r comfyui_controlnet_aux/requirements.txt

%cd /kaggle/working/ComfyUI
# IMPORTANTE: use sempre URLs /resolve/ (e NAO /blob/) do Hugging Face.
!find models -type f \( -name "*.safetensors" -o -name "*.ckpt" -o -name "*.pth" \) -size -1M -print -delete
!wget -c -q --show-progress https://huggingface.co/Lykon/DreamShaper/resolve/main/DreamShaper_8_pruned.safetensors -P ./models/checkpoints/
!wget -c -q --show-progress https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/
!wget -c -q --show-progress https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_openpose_fp16.safetensors -P ./models/controlnet/
!wget -c -q --show-progress https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_v11p_sd15_softedge_fp16.safetensors -P ./models/controlnet/
!ls -lh models/checkpoints models/vae models/controlnet


In [ ]:
# Inicia o ComfyUI em segundo plano (as proximas celulas continuam livres)
import subprocess, time, urllib.request
%cd /kaggle/working/ComfyUI
log = open('/kaggle/working/comfyui.log', 'w')
proc = subprocess.Popen(["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--cuda-device", "0", "--dont-print-server"], stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        print("ComfyUI pronto")
        break
    except Exception:
        if proc.poll() is not None:
            raise RuntimeError("ComfyUI encerrou; veja /kaggle/working/comfyui.log")
        time.sleep(2)
else:
    raise RuntimeError("Timeout; veja /kaggle/working/comfyui.log")


In [ ]:
# Extrai frames do video. FPS_EXTRACT baixo = menos imagens (8 ~ "anime em twos/threes")
import glob, os, shutil
VIDEO = None          # ex.: "/kaggle/input/meu-dataset/bautista.mp4" (None = primeiro .mp4 encontrado)
START_SEC = 0         # inicio do trecho (segundos)
DURATION_SEC = 4      # duracao do trecho
FPS_EXTRACT = 8

if VIDEO is None:
    found = glob.glob('/kaggle/input/**/*.mp4', recursive=True) + glob.glob('/kaggle/working/*.mp4')
    assert found, "Nenhum .mp4 encontrado: suba o video como Dataset"
    VIDEO = found[0]
print("Video:", VIDEO)
shutil.rmtree('/kaggle/working/frames_in', ignore_errors=True)
os.makedirs('/kaggle/working/frames_in')
!ffmpeg -loglevel error -ss {START_SEC} -t {DURATION_SEC} -i "{VIDEO}" -vf "fps={FPS_EXTRACT}" /kaggle/working/frames_in/%04d.png
print(len(os.listdir('/kaggle/working/frames_in')), "frames extraidos")


In [ ]:
# Transforma cada frame no ComfyUI usando a MESMA seed (menos oscilacao entre frames)
import json, time, uuid, shutil, os, glob, copy, urllib.request

WORKFLOW = "/kaggle/working/img2img_frame_api.json"   # suba workflows/img2img_frame_api.json no Kaggle
SEED = 675954591524960
DENOISE = 0.65            # mais baixo = mais fiel ao frame original
SOFTEDGE = 0.6
POSITIVE = None           # None = usa o prompt do workflow
NEGATIVE = None
ONLY_FRAMES = None        # ex.: [5, 6, 12] para refazer so esses (use outra SEED para variar)

API = "http://127.0.0.1:8188"
wf = json.load(open(WORKFLOW, encoding="utf8"))
wf["19"]["inputs"].update(seed=SEED, denoise=DENOISE)
wf["13"]["inputs"]["strength"] = SOFTEDGE
if POSITIVE:
    wf["6"]["inputs"]["text"] = POSITIVE
if NEGATIVE:
    wf["7"]["inputs"]["text"] = NEGATIVE

def post(path, data):
    req = urllib.request.Request(API + path, json.dumps(data).encode(), {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req))

frames = sorted(glob.glob('/kaggle/working/frames_in/*.png'))
os.makedirs('/kaggle/working/frames_out', exist_ok=True)
for path in frames:
    idx = int(os.path.basename(path)[:-4])
    if ONLY_FRAMES and idx not in ONLY_FRAMES:
        continue
    name = f"f{idx:04d}.png"
    shutil.copy(path, f"/kaggle/working/ComfyUI/input/{name}")
    p = copy.deepcopy(wf)
    p["1"]["inputs"]["image"] = name
    p["21"]["inputs"]["filename_prefix"] = f"out_{idx:04d}"
    pid = post("/prompt", {"prompt": p, "client_id": str(uuid.uuid4())})["prompt_id"]
    while True:
        h = json.load(urllib.request.urlopen(f"{API}/history/{pid}"))
        if pid in h:
            break
        time.sleep(1)
    entry = h[pid]
    if entry["status"]["status_str"] != "success":
        raise RuntimeError(f"Frame {idx} falhou: {entry['status']}")
    img = entry["outputs"]["21"]["images"][0]
    src = os.path.join("/kaggle/working/ComfyUI/output", img.get("subfolder", ""), img["filename"])
    shutil.copy(src, f"/kaggle/working/frames_out/{idx:04d}.png")
    print("frame", idx, "ok")


In [ ]:
# Monta o video. Se existir /kaggle/working/frames_edited, ele tem prioridade sobre frames_out
import os, shutil
FPS_ANIM = 8      # velocidade da animacao (igual ao FPS_EXTRACT = tempo real)
FPS_OUT = 24      # fps do arquivo final (frames repetidos; compativel com Shorts)

merged = '/kaggle/working/frames_final'
shutil.rmtree(merged, ignore_errors=True)
shutil.copytree('/kaggle/working/frames_out', merged)
ed = '/kaggle/working/frames_edited'
if os.path.isdir(ed):
    for f in os.listdir(ed):
        shutil.copy(os.path.join(ed, f), os.path.join(merged, f))
        print("editado:", f)
!ffmpeg -y -loglevel error -framerate {FPS_ANIM} -i {merged}/%04d.png -r {FPS_OUT} -c:v libx264 -pix_fmt yuv420p -crf 18 /kaggle/working/animacao.mp4
print("Pronto: /kaggle/working/animacao.mp4")
